# 02. Train an adapter and evaluate it on the gold set (Phase 5 onward)
Runtime → Change runtime type → **T4 GPU**.

**Training is resumable.** The adapter folder lives on Drive and a checkpoint is saved every 25 steps. If Colab disconnects, reconnect and rerun cells 1, 2, 3, 4: training continues from the last checkpoint. (Push `src/train_qwen.py` first: it needs the `--save_steps` and `--resume` options.)

Run in order: **1, 2, 3, 4, 5, 6, 8, 9, 10, 11**. Cell 7 only after a runtime reset (run 1 and 2 first, then 7, then continue from 8).

Training data: `data/synthetic_v2/` (codebook rules, zero overlap with the gold set). The baseline data and the baseline adapter are never touched. The adapter is `checkpoints/qwen25_3b_cpe_lora_v1`, backed up to Drive in `MyDrive/lora_projects/cpe_certificate_extraction/adapters`.

**Next versions:** change only `RUN` and `TRAIN_DIR` in cell 2 (for example `RUN = "v2"`, `TRAIN_DIR = "data/synthetic_v3"`), then run the same cells. Every version gets its own adapter folder and its own result files, and cell 10 compares it with the baseline.

In [ ]:
# 1. WHAT IT DOES: clones the project repo (or pulls the latest commit) and installs dependencies. Run it first in every new session.
# PHASE: every notebook starts with this.
# 1. Clone (or update) the repo and install dependencies
REPO_URL = "https://github.com/m-mirkan/peft-cpe-extraction-pipeline.git"
BRANCH = "feat/codebook-data-curation"

import os
if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt gdown

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 2. WHAT IT DOES: mounts Google Drive and sets the names used by every later cell (version tag, data folder, adapter folder, result files).
# To train a new version later, change only RUN and TRAIN_DIR below.
# Drive layout:  MyDrive/lora_projects/cpe_certificate_extraction/{adapters, results}
from google.colab import drive
drive.mount('/content/drive')

RUN = "v1"                                   # version tag of this experiment
TRAIN_DIR = "data/synthetic_v2"              # folder with synthetic_train.jsonl / synthetic_val.jsonl (never data/synthetic)
ADAPTER_NAME = f"qwen25_3b_cpe_lora_{RUN}"
ADAPTER_DIR = f"checkpoints/{ADAPTER_NAME}"  # local folder (the baseline adapter is checkpoints/qwen25_3b_cpe_lora, never use it here)
OUT_P1 = f"outputs/gold_{RUN}_p1.json"
OUT_P2 = f"outputs/gold_{RUN}_p2.json"
OUT_SCORES = f"outputs/gold_{RUN}_scores.json"

DRIVE_PROJECT = "/content/drive/MyDrive/lora_projects/cpe_certificate_extraction"
DRIVE_ADAPTERS = DRIVE_PROJECT + "/adapters"
DRIVE_RESULTS = DRIVE_PROJECT + "/results"
DRIVE_ADAPTER_DIR = f"{DRIVE_ADAPTERS}/{ADAPTER_NAME}"   # training writes here (checkpoints + final adapter)
# Same as the training defaults: batch 2 x gradient accumulation 8 = effective batch 16. Do not change these between runs you want to compare.
BATCH_SIZE, GRAD_ACCUM = 2, 8
!mkdir -p "{DRIVE_ADAPTERS}" "{DRIVE_RESULTS}"
assert ADAPTER_NAME != "qwen25_3b_cpe_lora", "refusing: that is the baseline adapter name"
print("run:", RUN, "| data:", TRAIN_DIR, "| adapter:", ADAPTER_DIR)
print("Drive folders ready:", DRIVE_ADAPTERS, "|", DRIVE_RESULTS)

In [ ]:
# 3. WHAT IT DOES: sanity check before training. Confirms the training files and the gold files are in the repo and shows the record counts.
import os, json
for p in [TRAIN_DIR + "/synthetic_train.jsonl", TRAIN_DIR + "/synthetic_val.jsonl",
          "data/gold/gold_inputs.json", "data/gold/gold_template.json"]:
    assert os.path.exists(p), "MISSING (did you push and pull?): " + p
assert os.path.normpath(TRAIN_DIR) != os.path.normpath("data/synthetic"), "refusing: that is the baseline training data"
n_tr = sum(1 for _ in open(TRAIN_DIR + "/synthetic_train.jsonl", encoding="utf-8"))
n_va = sum(1 for _ in open(TRAIN_DIR + "/synthetic_val.jsonl", encoding="utf-8"))
print("train records:", n_tr, "| val records:", n_va, "(v2 has 2200 / 200)")
print("first record input:", json.loads(open(TRAIN_DIR + "/synthetic_train.jsonl", encoding="utf-8").readline())["input"])
print("Training output folder:", ADAPTER_DIR)
!grep -c "save_steps" src/train_qwen.py   # must print a number >= 1 (patched trainer with resumable checkpoints)

In [ ]:
# 4. WHAT IT DOES: trains a LoRA adapter on the data in TRAIN_DIR. Same training settings as the baseline
# (2 epochs, lr 2e-4, LoRA r=16, batch 2 x accumulation 8). Writes straight to Google Drive and saves a checkpoint every 25 steps.
# If Colab disconnects: reconnect, run cells 1, 2, 3, then this cell again. It resumes from the last checkpoint.
# The adapter itself is written at the very end. Keep the last train/validation loss lines for the report.
!python src/train_qwen.py \
  --train_file {TRAIN_DIR}/synthetic_train.jsonl \
  --val_file {TRAIN_DIR}/synthetic_val.jsonl \
  --output_dir {DRIVE_ADAPTER_DIR} \
  --batch_size {BATCH_SIZE} --grad_accum {GRAD_ACCUM} \
  --save_steps 25 --resume

In [ ]:
# 5. WHAT IT DOES: after training has FINISHED, copies the final adapter files from Drive to checkpoints/ (fast local disk for inference)
# and makes a zip next to it on Drive. The intermediate checkpoint-* folders are not copied.
import os, shutil
assert os.path.exists(DRIVE_ADAPTER_DIR + "/adapter_model.safetensors"), "final adapter not found on Drive: training (cell 4) has not finished"
os.makedirs(ADAPTER_DIR, exist_ok=True)
for f in os.listdir(DRIVE_ADAPTER_DIR):
    src = os.path.join(DRIVE_ADAPTER_DIR, f)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join(ADAPTER_DIR, f))
!cd checkpoints && zip -r -q "{DRIVE_ADAPTERS}/{ADAPTER_NAME}.zip" {ADAPTER_NAME}
print("Local adapter:", sorted(os.listdir(ADAPTER_DIR)))
print("Drive:")
!ls -la "{DRIVE_ADAPTERS}"
print("Optional: once everything works you can delete the checkpoint-* folders inside", DRIVE_ADAPTER_DIR)

In [ ]:
# 6. WHAT IT DOES: lists the adapter files to confirm the adapter is complete (adapter_model.safetensors + adapter_config.json).
print(sorted(os.listdir(ADAPTER_DIR)))

In [ ]:
# 7. WHAT IT DOES: RESTORES the finished adapter from Drive. Run this ONLY in a new session / after a runtime reset
# (run cells 1 and 2 first). Skips the copy if the adapter is already in checkpoints/.
import os, shutil
if not os.path.exists(ADAPTER_DIR + "/adapter_config.json"):
    os.makedirs(ADAPTER_DIR, exist_ok=True)
    for f in os.listdir(DRIVE_ADAPTER_DIR):
        src = os.path.join(DRIVE_ADAPTER_DIR, f)
        if os.path.isfile(src):
            shutil.copy(src, os.path.join(ADAPTER_DIR, f))
print(sorted(os.listdir(ADAPTER_DIR)))

In [ ]:
# 8. WHAT IT DOES: runs the adapter on gold titles 1-50. Predictions are saved after every title
# to outputs/gold_<RUN>_p1.json (new file, the baseline files are not overwritten).
!python src/infer_and_resolve.py --lora_dir {ADAPTER_DIR} --input_file data/gold/gold_inputs.json --offset 0 --limit 50 --output_file {OUT_P1}

In [ ]:
# 9. WHAT IT DOES: runs the adapter on gold titles 51-100 -> outputs/gold_<RUN>_p2.json.
!python src/infer_and_resolve.py --lora_dir {ADAPTER_DIR} --input_file data/gold/gold_inputs.json --offset 50 --limit 50 --output_file {OUT_P2}

In [ ]:
# 10. WHAT IT DOES: scores the adapter on the gold set and, if the baseline files are in the repo
# (results/baseline_v0/), prints both models side by side (same scorer, same 100 titles).
import json, os
n1 = len(json.load(open(OUT_P1, encoding="utf-8")))
n2 = len(json.load(open(OUT_P2, encoding="utf-8")))
print(RUN, "predictions saved:", n1, "+", n2, "(expected 50 + 50)")
cmd = f"python scripts/score_gold.py --gold data/gold/gold_template.json --pred {RUN}={OUT_P1},{OUT_P2}"
b1, b2 = "results/baseline_v0/gold_base_p1.json", "results/baseline_v0/gold_base_p2.json"
if os.path.exists(b1) and os.path.exists(b2):
    cmd += f" --pred baseline={b1},{b2}"
else:
    print("baseline files not found in results/baseline_v0/, scoring this run only")
cmd += f" --json_out {OUT_SCORES}"
!{cmd}

In [ ]:
# 11. WHAT IT DOES: backs up this run's predictions and scores to Drive, then downloads them to your computer.
# Keep the two prediction files and the scores file: they are the input for the comparison and the report.
!cp {OUT_P1} {OUT_P2} {OUT_SCORES} "{DRIVE_RESULTS}/"
from google.colab import files
for f in [OUT_P1, OUT_P2, OUT_SCORES]:
    files.download(f)